# CPL Crop Disease Detection Model - Online Training

Dataset path:

`/home/jupyter/combined_plant_dataset_normalized/images`

In [1]:
from pathlib import Path

DATASET_DIR = Path('/home/jupyter/combined_plant_dataset_normalized/images')
CACHE_DIR = Path('/home/jupyter/cpl_hackathon')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

image_paths = sorted(
    path for path in DATASET_DIR.rglob('*')
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

print(f'Dataset directory: {DATASET_DIR}')
print(f'Cache directory: {CACHE_DIR}')
print(f'Total images found: {len(image_paths):,}')

Dataset directory: /home/jupyter/combined_plant_dataset_normalized/images
Cache directory: /home/jupyter/cpl_hackathon
Total images found: 119,059


In [2]:
import pandas as pd
from PIL import Image, UnidentifiedImageError

MIN_IMAGES_PER_CLASS = 10
CLEAN_MANIFEST_PATH = CACHE_DIR / 'cpl_clean_manifest.csv'
REJECTED_FILES_PATH = CACHE_DIR / 'cpl_rejected_files.csv'

def extract_labels(image_path):
    relative_path = image_path.relative_to(DATASET_DIR)
    parts = relative_path.parts

    if len(parts) < 3:
        return None

    crop_name = parts[0]
    health_group = parts[1].lower()

    if health_group in {'healthy', 'diseased'} and len(parts) >= 4:
        disease_name = parts[2]
    else:
        disease_name = parts[1]

    crop_disease_label = f'{crop_name}::{disease_name}'
    return crop_name, disease_name, crop_disease_label

if CLEAN_MANIFEST_PATH.exists():
    clean_df = pd.read_csv(CLEAN_MANIFEST_PATH)
    bad_files_df = pd.read_csv(REJECTED_FILES_PATH) if REJECTED_FILES_PATH.exists() else pd.DataFrame()

    label_names = sorted(clean_df['crop_disease_label'].unique())
    label_to_id = {label: index for index, label in enumerate(label_names)}
    id_to_label = {index: label for label, index in label_to_id.items()}
    clean_df['label_id'] = clean_df['crop_disease_label'].map(label_to_id).astype(int)

    print('Loaded cached clean manifest')
else:
    clean_rows = []
    bad_files = []

    for image_path in image_paths:
        labels = extract_labels(image_path)
        if labels is None:
            bad_files.append({'image_path': str(image_path), 'reason': 'invalid_folder_structure'})
            continue

        try:
            with Image.open(image_path) as image:
                image.verify()
                width, height = image.size
        except (OSError, UnidentifiedImageError):
            bad_files.append({'image_path': str(image_path), 'reason': 'unreadable_image'})
            continue

        if width < 64 or height < 64:
            bad_files.append({'image_path': str(image_path), 'reason': 'image_too_small'})
            continue

        crop_name, disease_name, crop_disease_label = labels
        clean_rows.append({
            'image_path': str(image_path),
            'crop_name': crop_name,
            'disease_name': disease_name,
            'crop_disease_label': crop_disease_label,
            'width': width,
            'height': height,
        })

    clean_df = pd.DataFrame(clean_rows)
    bad_files_df = pd.DataFrame(bad_files)

    class_counts = clean_df['crop_disease_label'].value_counts()
    valid_classes = class_counts[class_counts >= MIN_IMAGES_PER_CLASS].index
    clean_df = clean_df[clean_df['crop_disease_label'].isin(valid_classes)].reset_index(drop=True)

    label_names = sorted(clean_df['crop_disease_label'].unique())
    label_to_id = {label: index for index, label in enumerate(label_names)}
    id_to_label = {index: label for label, index in label_to_id.items()}
    clean_df['label_id'] = clean_df['crop_disease_label'].map(label_to_id).astype(int)

    clean_df.to_csv(CLEAN_MANIFEST_PATH, index=False)
    bad_files_df.to_csv(REJECTED_FILES_PATH, index=False)
    print('Created and cached clean manifest')

print(f'Clean images: {len(clean_df):,}')
print(f'Rejected files: {len(bad_files_df):,}')
print(f'Classes kept: {len(label_names):,}')

Loaded cached clean manifest
Clean images: 119,009
Rejected files: 36
Classes kept: 139


In [3]:
MAX_IMAGES_PER_CLASS = 800
BALANCED_MANIFEST_PATH = CACHE_DIR / 'cpl_balanced_manifest.csv'
CLASS_BALANCE_SUMMARY_PATH = CACHE_DIR / 'cpl_class_balance_summary.csv'

if BALANCED_MANIFEST_PATH.exists():
    balanced_df = pd.read_csv(BALANCED_MANIFEST_PATH)
    balance_summary_df = pd.read_csv(CLASS_BALANCE_SUMMARY_PATH) if CLASS_BALANCE_SUMMARY_PATH.exists() else pd.DataFrame()

    label_names = sorted(balanced_df['crop_disease_label'].unique())
    label_to_id = {label: index for index, label in enumerate(label_names)}
    id_to_label = {index: label for label, index in label_to_id.items()}
    balanced_df['label_id'] = balanced_df['crop_disease_label'].map(label_to_id).astype(int)

    print('Loaded cached balanced manifest')
else:
    before_counts = clean_df['crop_disease_label'].value_counts().rename('before_count')

    # Sample per class without using groupby().apply() so we avoid the
    # pandas 2.2 FutureWarning about grouping columns being passed to apply.
    sampled_groups = [
        group.sample(n=min(len(group), MAX_IMAGES_PER_CLASS), random_state=42)
        for _, group in clean_df.groupby('crop_disease_label')
    ]
    balanced_df = (
        pd.concat(sampled_groups, ignore_index=True)
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    label_names = sorted(balanced_df['crop_disease_label'].unique())
    label_to_id = {label: index for index, label in enumerate(label_names)}
    id_to_label = {index: label for label, index in label_to_id.items()}
    balanced_df['label_id'] = balanced_df['crop_disease_label'].map(label_to_id).astype(int)

    after_counts = balanced_df['crop_disease_label'].value_counts().rename('after_count')
    balance_summary_df = (
        pd.concat([before_counts, after_counts], axis=1)
        .fillna(0)
        .astype(int)
        .reset_index()
        .rename(columns={'index': 'crop_disease_label'})
        .sort_values('before_count', ascending=False)
    )

    balanced_df.to_csv(BALANCED_MANIFEST_PATH, index=False)
    balance_summary_df.to_csv(CLASS_BALANCE_SUMMARY_PATH, index=False)
    print('Created and cached balanced manifest')

print(f'Balanced images: {len(balanced_df):,}')
print(f'Balanced classes: {balanced_df["crop_disease_label"].nunique():,}')
print(f'Max images per class: {MAX_IMAGES_PER_CLASS:,}')
balance_summary_df.head(20)

Loaded cached balanced manifest
Balanced images: 65,341
Balanced classes: 139
Max images per class: 800


,crop_disease_label,before_count,after_count
0,cabbage::Alternaria leaf spot,4370,800
1,cabbage::Healthy,4288,800
2,maize::Maize grasshoper,3712,800
3,maize::Maize leaf beetle,3616,800
4,maize::Maize healthy,3520,800
5,maize::Maize leaf spot,3485,800
6,maize::Maize streak virus,3472,800
7,onion::Healthy leaves,3436,800
8,maize::Maize leaf blight,3376,800
9,maize::Maize Lethal Necrosis,3231,800


In [4]:
import json
from sklearn.model_selection import train_test_split

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SPLIT_RANDOM_STATE = 42

TRAIN_MANIFEST_PATH = CACHE_DIR / 'cpl_train_manifest.csv'
VAL_MANIFEST_PATH = CACHE_DIR / 'cpl_val_manifest.csv'
TEST_MANIFEST_PATH = CACHE_DIR / 'cpl_test_manifest.csv'
SPLIT_SUMMARY_PATH = CACHE_DIR / 'cpl_split_summary.csv'
LABEL_TO_ID_PATH = CACHE_DIR / 'cpl_label_to_id.json'
ID_TO_LABEL_PATH = CACHE_DIR / 'cpl_id_to_label.json'

split_cache_exists = all(path.exists() for path in [
    TRAIN_MANIFEST_PATH,
    VAL_MANIFEST_PATH,
    TEST_MANIFEST_PATH,
    SPLIT_SUMMARY_PATH,
    LABEL_TO_ID_PATH,
    ID_TO_LABEL_PATH,
])

if split_cache_exists:
    train_df = pd.read_csv(TRAIN_MANIFEST_PATH)
    val_df = pd.read_csv(VAL_MANIFEST_PATH)
    test_df = pd.read_csv(TEST_MANIFEST_PATH)
    split_summary_df = pd.read_csv(SPLIT_SUMMARY_PATH)

    with open(LABEL_TO_ID_PATH, 'r') as file:
        label_to_id = json.load(file)
    with open(ID_TO_LABEL_PATH, 'r') as file:
        id_to_label = {int(key): value for key, value in json.load(file).items()}

    print('Loaded cached train/validation/test split')
else:
    label_names = sorted(balanced_df['crop_disease_label'].unique())
    label_to_id = {label: index for index, label in enumerate(label_names)}
    id_to_label = {index: label for label, index in label_to_id.items()}

    split_df = balanced_df.copy()
    split_df['label_id'] = split_df['crop_disease_label'].map(label_to_id).astype(int)

    train_df, temp_df = train_test_split(
        split_df,
        train_size=TRAIN_RATIO,
        stratify=split_df['label_id'],
        random_state=SPLIT_RANDOM_STATE,
        shuffle=True,
    )

    test_size_from_temp = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=test_size_from_temp,
        stratify=temp_df['label_id'],
        random_state=SPLIT_RANDOM_STATE,
        shuffle=True,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    split_summary_df = (
        pd.concat([
            train_df['crop_disease_label'].value_counts().rename('train_count'),
            val_df['crop_disease_label'].value_counts().rename('val_count'),
            test_df['crop_disease_label'].value_counts().rename('test_count'),
        ], axis=1)
        .fillna(0)
        .astype(int)
        .reset_index()
        .rename(columns={'index': 'crop_disease_label'})
    )
    split_summary_df['total_count'] = split_summary_df[['train_count', 'val_count', 'test_count']].sum(axis=1)
    split_summary_df = split_summary_df.sort_values('total_count', ascending=False).reset_index(drop=True)

    train_df.to_csv(TRAIN_MANIFEST_PATH, index=False)
    val_df.to_csv(VAL_MANIFEST_PATH, index=False)
    test_df.to_csv(TEST_MANIFEST_PATH, index=False)
    split_summary_df.to_csv(SPLIT_SUMMARY_PATH, index=False)

    with open(LABEL_TO_ID_PATH, 'w') as file:
        json.dump(label_to_id, file, indent=2)
    with open(ID_TO_LABEL_PATH, 'w') as file:
        json.dump(id_to_label, file, indent=2)

    print('Created and cached train/validation/test split')

print(f'Train images: {len(train_df):,} ({len(train_df) / len(balanced_df):.1%})')
print(f'Validation images: {len(val_df):,} ({len(val_df) / len(balanced_df):.1%})')
print(f'Test images: {len(test_df):,} ({len(test_df) / len(balanced_df):.1%})')
print(f'Classes: {len(label_to_id):,}')

split_summary_df.head(20)

Loaded cached train/validation/test split
Train images: 45,738 (70.0%)
Validation images: 9,800 (15.0%)
Test images: 9,802 (15.0%)
Classes: 139


,crop_disease_label,train_count,val_count,test_count,total_count
0,groundnut::healthy_leaf_1,560,120,120,800
1,soyabean::Soybean Healthy,560,120,120,800
2,maize::Maize leaf blight,560,120,120,800
3,tomato::Tomato___Septoria_leaf_spot,560,120,120,800
4,maize::Maize grasshoper,560,120,120,800
5,sorghum::Cereal Grain molds (White Fungi)t,560,120,120,800
6,tomato::Tomato___Target_Spot,560,120,120,800
7,tomato::Tomato___healthy,560,120,120,800
8,sorghum::Healthy Sorghum,560,120,120,800
9,groundnut::rust_1,560,120,120,800


In [5]:
import tensorflow as tf

# Reproducibility for shuffle / random-augmentation ops
tf.random.set_seed(SPLIT_RANDOM_STATE)

# Hardware sanity check — this notebook is intended to run on CPU.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPUs detected: {gpus}')
else:
    print('No GPU detected -> training will run on CPU. Expect long epoch times.')

MODEL_BACKBONE = 'efficientnetb2'
IMAGE_SIZE = (260, 260)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
USE_TF_DATA_DISK_CACHE = False

PREPROCESSING_CONFIG_PATH = CACHE_DIR / 'cpl_preprocessing_config.json'

preprocessing_config = {
    'model_backbone': MODEL_BACKBONE,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'use_tf_data_disk_cache': USE_TF_DATA_DISK_CACHE,
}

with open(PREPROCESSING_CONFIG_PATH, 'w') as file:
    json.dump(preprocessing_config, file, indent=2)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name='data_augmentation')

def decode_and_resize(image_path, label_id):
    image_bytes = tf.io.read_file(image_path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMAGE_SIZE, method='bilinear')
    image = tf.cast(image, tf.float32)
    label_id = tf.cast(label_id, tf.int32)
    return image, label_id

def apply_training_augmentation(image, label_id):
    image = data_augmentation(image, training=True)
    return image, label_id

def build_dataset(dataframe, training=False, cache_name=None):
    image_paths_tensor = dataframe['image_path'].astype(str).values
    labels_tensor = dataframe['label_id'].astype('int32').values

    dataset = tf.data.Dataset.from_tensor_slices((image_paths_tensor, labels_tensor))

    if training:
        dataset = dataset.shuffle(buffer_size=min(len(dataframe), 10000), seed=SPLIT_RANDOM_STATE, reshuffle_each_iteration=True)

    dataset = dataset.map(decode_and_resize, num_parallel_calls=AUTOTUNE)

    if USE_TF_DATA_DISK_CACHE and cache_name is not None:
        dataset = dataset.cache(str(CACHE_DIR / cache_name))

    if training:
        dataset = dataset.map(apply_training_augmentation, num_parallel_calls=AUTOTUNE)

    dataset = dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return dataset

train_ds = build_dataset(train_df, training=True, cache_name='tf_train_cache')
val_ds = build_dataset(val_df, training=False, cache_name='tf_val_cache')
test_ds = build_dataset(test_df, training=False, cache_name='tf_test_cache')

num_classes = len(label_to_id)

print(f'Model backbone: {MODEL_BACKBONE}')
print(f'Image size: {IMAGE_SIZE}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Number of classes: {num_classes:,}')

2026-05-28 02:43:06.224056: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-28 02:43:07.096237: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/lib/x86_64-linux-gnu/:/opt/conda/lib
2026-05-28 02:43:07.096357: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local

No GPU detected -> training will run on CPU. Expect long epoch times.


2026-05-28 02:43:07.616483: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/lib/x86_64-linux-gnu/:/opt/conda/lib
2026-05-28 02:43:07.616523: W tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:265] failed call to cuInit: UNKNOWN ERROR (303)
2026-05-28 02:43:07.616544: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (instance-20260414-172140): /proc/driver/nvidia/version does not exist
2026-05-28 02:43:07.624341: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable the

Model backbone: efficientnetb2
Image size: (260, 260)
Batch size: 32
Number of classes: 139


## Data fix applied during training

During Phase 1, one corrupted `.webp` file was discovered in the validation set:
`combined_plant_dataset_normalized/images/cauliflower/diseased/Bacterial Soft Rot/bacterial soft rot10__dup1__dup1.webp`

`PIL.Image.verify()` (used in the manifest cell) accepts WEBP, but TensorFlow 2.11's `tf.io.decode_image` does not. The file was removed from the validation manifest and the change was persisted to `cpl_val_manifest.csv`, so re-running the data-loading cells above produces the corrected validation set automatically. No additional action is required at runtime.

In [ ]:
import re
from tensorflow.keras import layers, models

TOTAL_EPOCHS = 20
LEARNING_RATE = 1e-3
DROPOUT_RATE = 0.35

MODEL_DIR = CACHE_DIR / 'models'
EPOCH_CHECKPOINT_DIR = MODEL_DIR / 'epoch_checkpoints'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EPOCH_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# NOTE: TF 2.11 + EfficientNetB2 cannot save the full model (the built-in
# Normalization layer carries tensor-typed config that breaks JSON serialization
# in the .keras / .h5 / SavedModel paths; fixed in TF 2.12+). We therefore use
# WEIGHTS-ONLY checkpoints. Resume rebuilds the architecture and loads the
# weights, which is robust and produces identical inference results.
BEST_WEIGHTS_PATH = MODEL_DIR / 'cpl_best_efficientnetb2.weights.h5'
FINAL_WEIGHTS_PATH = MODEL_DIR / 'cpl_final_efficientnetb2.weights.h5'
TRAINING_HISTORY_PATH = CACHE_DIR / 'cpl_training_history.csv'
TEST_METRICS_PATH = CACHE_DIR / 'cpl_test_metrics.json'
ARCH_METADATA_PATH = MODEL_DIR / 'cpl_architecture_metadata.json'


def build_model(num_classes):
    """Re-create the exact architecture used for training; used for both
    fresh starts and for rebuilding before load_weights()."""
    base_model = tf.keras.applications.EfficientNetB2(
        include_top=False,
        weights='imagenet',
        input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
    )
    base_model.trainable = False

    inputs = layers.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), name='image')
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='global_average_pooling')(x)
    x = layers.BatchNormalization(name='head_batch_norm')(x)
    x = layers.Dropout(DROPOUT_RATE, name='head_dropout')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='crop_disease_prediction')(x)

    model = models.Model(inputs=inputs, outputs=outputs, name='cpl_efficientnetb2_crop_disease_model')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_accuracy'),
        ],
    )
    return model


def find_latest_epoch_checkpoint(checkpoint_dir):
    """Find the highest-epoch weights file so training can resume after a crash."""
    pattern = re.compile(r'epoch_(\d+)_val_acc_[\d.]+\.weights\.h5$')
    candidates = []
    for path in checkpoint_dir.glob('epoch_*.weights.h5'):
        match = pattern.search(path.name)
        if match:
            candidates.append((int(match.group(1)), path))
    if not candidates:
        return None, 0
    last_epoch, last_path = max(candidates, key=lambda item: item[0])
    return last_path, last_epoch


print('Building model architecture...')
model = build_model(num_classes)

resume_path, completed_epochs = find_latest_epoch_checkpoint(EPOCH_CHECKPOINT_DIR)
if resume_path is not None and completed_epochs < TOTAL_EPOCHS:
    print(f'Resuming weights from {resume_path.name} (completed epochs: {completed_epochs})')
    model.load_weights(resume_path)
    initial_epoch = completed_epochs
elif resume_path is not None and completed_epochs >= TOTAL_EPOCHS:
    print(f'All {TOTAL_EPOCHS} epochs already trained -> loading last weights for evaluation only')
    model.load_weights(resume_path)
    initial_epoch = TOTAL_EPOCHS
else:
    print('No checkpoint found -> starting from ImageNet pretraining')
    initial_epoch = 0

# Persist architecture metadata so the model can be rebuilt for inference
# without re-running the notebook.
with open(ARCH_METADATA_PATH, 'w') as file:
    json.dump({
        'backbone': 'EfficientNetB2',
        'image_size': list(IMAGE_SIZE),
        'num_classes': num_classes,
        'dropout_rate': DROPOUT_RATE,
        'preprocessing': 'EfficientNet built-in (expects [0,255] float32)',
    }, file, indent=2)

train_class_counts = train_df['label_id'].value_counts().sort_index()
class_weight = {
    int(class_id): float(len(train_df) / (num_classes * count))
    for class_id, count in train_class_counts.items()
}


class EpochPercentageLogger(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        percent_complete = ((epoch + 1) / TOTAL_EPOCHS) * 100
        logs = logs or {}
        print(
            f'Epoch {epoch + 1}/{TOTAL_EPOCHS} complete - '
            f'{percent_complete:.1f}% training done - '
            f'val_accuracy={logs.get("val_accuracy", 0):.4f} - '
            f'val_top_3_accuracy={logs.get("val_top_3_accuracy", 0):.4f}'
        )


callbacks = [
    # Per-epoch weights snapshot (kept on purpose for crash recovery)
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(EPOCH_CHECKPOINT_DIR / 'epoch_{epoch:02d}_val_acc_{val_accuracy:.4f}.weights.h5'),
        monitor='val_accuracy',
        save_best_only=False,
        save_weights_only=True,
        verbose=1,
    ),
    # Best-so-far weights, used later for test evaluation
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_WEIGHTS_PATH),
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(str(TRAINING_HISTORY_PATH), append=True),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    EpochPercentageLogger(),
]

if initial_epoch < TOTAL_EPOCHS:
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=TOTAL_EPOCHS,
        initial_epoch=initial_epoch,
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=1,
    )

    model.save_weights(FINAL_WEIGHTS_PATH)
    print(f'Final weights saved to: {FINAL_WEIGHTS_PATH}')
    print(f'Best weights saved to: {BEST_WEIGHTS_PATH}')
    print(f'Per-epoch weights saved in: {EPOCH_CHECKPOINT_DIR}')
else:
    print('Skipping training (all epochs already completed).')

# --- Test-set evaluation using the best validation snapshot ---
print('\nEvaluating on test set with best weights...')
eval_model = build_model(num_classes)
if BEST_WEIGHTS_PATH.exists():
    eval_model.load_weights(BEST_WEIGHTS_PATH)
    print(f'(loaded best weights from {BEST_WEIGHTS_PATH.name})')
else:
    eval_model = model
    print('(best weights not found, evaluating current in-memory model)')

test_metrics = eval_model.evaluate(test_ds, verbose=1, return_dict=True)
print(f'Test results: {test_metrics}')

with open(TEST_METRICS_PATH, 'w') as file:
    json.dump({k: float(v) for k, v in test_metrics.items()}, file, indent=2)
print(f'Test metrics saved to:    {TEST_METRICS_PATH}')
print(f'Architecture metadata at: {ARCH_METADATA_PATH}')

Building model architecture...
All 20 epochs already trained -> loading last weights for evaluation only
Skipping training (all epochs already completed).

Evaluating on test set with best weights...
(loaded best weights from cpl_best_efficientnetb2.weights.h5)
 13/307 [>.............................] - ETA: 3:59 - loss: 0.2942 - accuracy: 0.9159 - top_3_accuracy: 0.9880

## Phase 2: Fine-tuning

Phase 1 trained the head only with the EfficientNetB2 backbone frozen, reaching
**90.75% test accuracy / 99.10% top-3**. In this phase we unfreeze the **top 31
layers** of the backbone (all of `block7` plus `top_conv` / `top_bn` /
`top_activation`) and train at a much lower learning rate (`1e-5`, 100x lower)
to gently adapt those late features to plant-disease textures.

Key safeguards:
- **BatchNormalization layers stay frozen** even when their parent block is
  unfrozen (small-batch BN updates would destabilize the pretrained statistics).
- **`build_model()` calls the backbone with `training=False`**, so BN layers
  always use ImageNet moving stats, both before and during fine-tuning.
- The original phase-1 weights file is **not overwritten** — fine-tuned outputs
  use new filenames suffixed `_finetuned`. If fine-tuning hurts, you can fall
  back to the phase-1 best.
- The cell **resumes from the latest fine-tune checkpoint** if it crashes.

**Expected cost on CPU**: ~35–50 min/epoch (more trainable parameters than
phase 1). With `EarlyStopping(patience=3)`, total wall-clock is typically
**3–5 hours**. Realistic accuracy gain: **+1 to +3 percentage points**.

If your kernel was restarted after phase 1, re-run cells 1–5 first (they hit
caches and finish in seconds) so `train_ds`, `val_ds`, `test_ds`,
`build_model`, etc. are back in scope.

In [8]:
# Sanity check that previous cells have been run
required_names = ['build_model', 'num_classes', 'BEST_WEIGHTS_PATH', 'MODEL_DIR',
                  'CACHE_DIR', 'class_weight', 'train_ds', 'val_ds', 'test_ds',
                  'TEST_METRICS_PATH']
_missing = [n for n in required_names if n not in globals()]
if _missing:
    raise NameError(
        f'Missing variables {_missing}. Re-run cells 1-5 (and the training cell '
        f'to define build_model) before running fine-tuning.'
    )

# Fine-tuning configuration
FT_EPOCHS = 8
FT_LR = 1e-5                  # 100x smaller than phase-1 head training (1e-3)
FT_UNFREEZE_LAYERS = 31       # last 31 layers of EfficientNetB2 = all of block7 + top_conv

FT_CHECKPOINT_DIR = MODEL_DIR / 'finetune_checkpoints'
FT_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FT_BEST_WEIGHTS_PATH = MODEL_DIR / 'cpl_best_efficientnetb2_finetuned.weights.h5'
FT_FINAL_WEIGHTS_PATH = MODEL_DIR / 'cpl_final_efficientnetb2_finetuned.weights.h5'
FT_HISTORY_PATH = CACHE_DIR / 'cpl_finetune_history.csv'
FT_TEST_METRICS_PATH = CACHE_DIR / 'cpl_finetune_test_metrics.json'

# 1) Build a fresh model and load the best phase-1 weights as the starting point.
print('Building model and loading phase-1 best weights ...')
ft_model = build_model(num_classes)
ft_model.load_weights(BEST_WEIGHTS_PATH)
print(f'  loaded {BEST_WEIGHTS_PATH.name}')

# 2) Locate the EfficientNetB2 backbone inside the wrapper Model.
backbone = None
for layer in ft_model.layers:
    if 'efficientnet' in layer.name.lower():
        backbone = layer
        break
if backbone is None:
    raise RuntimeError('Could not find EfficientNet backbone inside ft_model')
print(f'Backbone "{backbone.name}" has {len(backbone.layers)} layers')

# 3) Unfreeze the last FT_UNFREEZE_LAYERS layers, but KEEP BatchNorm frozen.
backbone.trainable = True
n_frozen = len(backbone.layers) - FT_UNFREEZE_LAYERS
n_bn_frozen = 0
for i, layer in enumerate(backbone.layers):
    if i < n_frozen:
        layer.trainable = False
    elif isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
        n_bn_frozen += 1

unfrozen_non_bn = [l.name for l in backbone.layers[n_frozen:] if l.trainable]
print(f'Frozen layers (idx 0..{n_frozen-1}): {n_frozen}')
print(f'Unfrozen non-BN layers: {len(unfrozen_non_bn)} (of which BN kept frozen: {n_bn_frozen})')
print(f'First few unfrozen layers: {unfrozen_non_bn[:5]} ...')

# 4) Recompile with the much lower learning rate.
ft_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FT_LR),
    loss='sparse_categorical_crossentropy',
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_accuracy'),
    ],
)
trainable_params = sum(tf.keras.backend.count_params(w) for w in ft_model.trainable_weights)
total_params = ft_model.count_params()
print(f'Trainable params after unfreeze: {trainable_params:,} / {total_params:,} '
      f'({100*trainable_params/total_params:.1f}%)')


# 5) Resume logic for fine-tuning.
def find_latest_ft_checkpoint(checkpoint_dir):
    pattern = re.compile(r'ft_epoch_(\d+)_val_acc_[\d.]+\.weights\.h5$')
    candidates = []
    for path in checkpoint_dir.glob('ft_epoch_*.weights.h5'):
        m = pattern.search(path.name)
        if m:
            candidates.append((int(m.group(1)), path))
    if not candidates:
        return None, 0
    last_epoch, last_path = max(candidates, key=lambda kv: kv[0])
    return last_path, last_epoch


ft_resume_path, ft_completed = find_latest_ft_checkpoint(FT_CHECKPOINT_DIR)
if ft_resume_path is not None and ft_completed < FT_EPOCHS:
    print(f'Resuming fine-tuning from {ft_resume_path.name} (completed: {ft_completed})')
    ft_model.load_weights(ft_resume_path)
    ft_initial_epoch = ft_completed
elif ft_resume_path is not None and ft_completed >= FT_EPOCHS:
    print(f'Fine-tuning already complete -> loading last weights for evaluation only')
    ft_model.load_weights(ft_resume_path)
    ft_initial_epoch = FT_EPOCHS
else:
    print('Starting fine-tuning fresh from phase-1 best weights')
    ft_initial_epoch = 0

# 6) Callbacks (mirrors phase-1 design but with shorter patience).
ft_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(FT_CHECKPOINT_DIR / 'ft_epoch_{epoch:02d}_val_acc_{val_accuracy:.4f}.weights.h5'),
        monitor='val_accuracy',
        save_best_only=False,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(FT_BEST_WEIGHTS_PATH),
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(str(FT_HISTORY_PATH), append=True),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1,
    ),
]

# 7) Train.
if ft_initial_epoch < FT_EPOCHS:
    ft_history = ft_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FT_EPOCHS,
        initial_epoch=ft_initial_epoch,
        callbacks=ft_callbacks,
        class_weight=class_weight,
        verbose=1,
    )
    ft_model.save_weights(FT_FINAL_WEIGHTS_PATH)
    print(f'Fine-tuned final weights saved: {FT_FINAL_WEIGHTS_PATH}')
    print(f'Fine-tuned best weights saved:  {FT_BEST_WEIGHTS_PATH}')
else:
    print('Fine-tuning already done; skipping fit.')

# 8) Test-set evaluation with best fine-tuned weights.
print('\nEvaluating fine-tuned model on test set ...')
ft_eval_model = build_model(num_classes)

# The fine-tuned weights file was saved while the backbone had
# trainable=True with only the last FT_UNFREEZE_LAYERS layers unfrozen and
# all BatchNorm layers re-frozen. Keras saves weights in
# [trainable, then non_trainable] order, so to load them positionally we
# must replicate that exact trainable state on the fresh eval model
# before calling load_weights -- otherwise it raises
# 'ValueError: axes don't match array'.
_eval_backbone = next(
    l for l in ft_eval_model.layers if 'efficientnet' in l.name.lower()
)
_eval_backbone.trainable = True
_n_frozen_eval = len(_eval_backbone.layers) - FT_UNFREEZE_LAYERS
for _i, _layer in enumerate(_eval_backbone.layers):
    if _i < _n_frozen_eval:
        _layer.trainable = False
    elif isinstance(_layer, tf.keras.layers.BatchNormalization):
        _layer.trainable = False

if FT_BEST_WEIGHTS_PATH.exists():
    ft_eval_model.load_weights(FT_BEST_WEIGHTS_PATH)
    print(f'(loaded {FT_BEST_WEIGHTS_PATH.name})')
else:
    ft_eval_model = ft_model
    print('(best fine-tuned weights not found, evaluating current model in memory)')

ft_test_metrics = ft_eval_model.evaluate(test_ds, verbose=1, return_dict=True)
print(f'Fine-tuned test results: {ft_test_metrics}')

with open(FT_TEST_METRICS_PATH, 'w') as file:
    json.dump({k: float(v) for k, v in ft_test_metrics.items()}, file, indent=2)

# 9) Compare to phase 1.
with open(TEST_METRICS_PATH, 'r') as file:
    phase1_metrics = json.load(file)
print('\n=== Phase 1 vs Phase 2 (test set) ===')
print(f'  Phase 1 (frozen):     loss={phase1_metrics["loss"]:.4f}  acc={phase1_metrics["accuracy"]:.4f}  top3={phase1_metrics["top_3_accuracy"]:.4f}')
print(f'  Phase 2 (fine-tuned): loss={ft_test_metrics["loss"]:.4f}  acc={ft_test_metrics["accuracy"]:.4f}  top3={ft_test_metrics["top_3_accuracy"]:.4f}')
delta_acc = ft_test_metrics["accuracy"] - phase1_metrics["accuracy"]
delta_top3 = ft_test_metrics["top_3_accuracy"] - phase1_metrics["top_3_accuracy"]
print(f'  Delta:                acc={delta_acc:+.4f}  top3={delta_top3:+.4f}')
if delta_acc > 0:
    print('  -> Fine-tuning helped. Use cpl_best_efficientnetb2_finetuned.weights.h5 for inference.')
else:
    print('  -> Fine-tuning did not help. Stick with cpl_best_efficientnetb2.weights.h5.')

Building model and loading phase-1 best weights ...
  loaded cpl_best_efficientnetb2.weights.h5
Backbone "efficientnetb2" has 340 layers
Frozen layers (idx 0..308): 309
Unfrozen non-BN layers: 24 (of which BN kept frozen: 7)
First few unfrozen layers: ['block7a_expand_conv', 'block7a_expand_activation', 'block7a_dwconv', 'block7a_activation', 'block7a_se_squeeze'] ...
Trainable params after unfreeze: 3,415,255 / 7,970,052 (42.9%)
Fine-tuning already complete -> loading last weights for evaluation only
Fine-tuning already done; skipping fit.

Evaluating fine-tuned model on test set ...
(loaded cpl_best_efficientnetb2_finetuned.weights.h5)
307/307 [==============================] - 261s 844ms/step - loss: 0.2136 - accuracy: 0.9361 - top_3_accuracy: 0.9935
Fine-tuned test results: {'loss': 0.21360865235328674, 'accuracy': 0.936135470867157, 'top_3_accuracy': 0.9934707283973694}

=== Phase 1 vs Phase 2 (test set) ===
  Phase 1 (frozen):     loss=0.2939  acc=0.9075  top3=0.9910
  Phase 2 (f

## Phase 3: Comprehensive test-set metrics

Beyond the headline loss / accuracy / top-3 numbers reported above, this cell produces the full evaluation pack:

- **Per-class precision, recall, F1** via `sklearn.metrics.classification_report` (text + JSON).
- **Confusion matrix** (139×139, CSV).
- **Top-1 accuracy per class**, sorted ascending so the weak classes surface first.
- **Top-30 most-confused class pairs**, useful for diagnosing which mistakes are systematic vs random.
- **Raw probability tensor + image paths**, cached as `cpl_test_predictions.npz` so downstream visualisation scripts don’t need to re-run inference.

Outputs go to `cpl_hackathon/reports/`. Inference is run once on the in-memory `ft_eval_model` and reused for every metric. Expect ~6 minutes on CPU.

In [ ]:
# === Comprehensive test-set metrics ===
# Reuses ft_eval_model, test_ds, test_df, id_to_label, num_classes from above.
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

REPORTS_DIR = CACHE_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = REPORTS_DIR / 'cpl_test_predictions.npz'
REPORT_TXT = REPORTS_DIR / 'cpl_test_classification_report.txt'
REPORT_JSON = REPORTS_DIR / 'cpl_test_classification_report.json'
PER_CLASS_PATH = REPORTS_DIR / 'cpl_test_per_class_accuracy.csv'
CONF_MATRIX_PATH = REPORTS_DIR / 'cpl_test_confusion_matrix.csv'
CONFUSED_PAIRS_PATH = REPORTS_DIR / 'cpl_test_top_confused_pairs.csv'
TOP_PAIRS_N = 30

class_names = [id_to_label[i] for i in range(num_classes)]

# 1) Run inference once on the test pipeline.
print(f'Running inference on {len(test_df):,} test images ...')
probs = ft_eval_model.predict(test_ds, verbose=1)
y_true = test_df['label_id'].to_numpy()
y_pred = probs.argmax(axis=1)
top3_pred = np.argsort(-probs, axis=1)[:, :3]

# 2) Cache raw predictions for visualisation scripts.
np.savez_compressed(
    PREDICTIONS_PATH,
    probs=probs.astype(np.float32),
    y_true=y_true.astype(np.int32),
    y_pred=y_pred.astype(np.int32),
    top3_pred=top3_pred.astype(np.int32),
    image_paths=test_df['image_path'].astype(str).values,
)
print(f'Wrote {PREDICTIONS_PATH.relative_to(CACHE_DIR)}')

# 3) Classification report (text + JSON).
report_txt = classification_report(
    y_true, y_pred, labels=list(range(num_classes)),
    target_names=class_names, digits=4, zero_division=0,
)
REPORT_TXT.write_text(report_txt)
report_dict = classification_report(
    y_true, y_pred, labels=list(range(num_classes)),
    target_names=class_names, digits=4, zero_division=0,
    output_dict=True,
)
with open(REPORT_JSON, 'w') as fh:
    json.dump(report_dict, fh, indent=2)
print(f'Wrote {REPORT_TXT.relative_to(CACHE_DIR)}')
print(f'Wrote {REPORT_JSON.relative_to(CACHE_DIR)}')

# 4) Per-class top-1 accuracy (sorted ascending so the weak classes are first).
per_class_rows = []
for cls_id in range(num_classes):
    mask = y_true == cls_id
    support = int(mask.sum())
    acc = float((y_pred[mask] == cls_id).mean()) if support else float('nan')
    per_class_rows.append({
        'label_id': cls_id,
        'crop_disease_label': id_to_label[cls_id],
        'support': support,
        'top1_accuracy': acc,
    })
per_class_df = pd.DataFrame(per_class_rows).sort_values('top1_accuracy', na_position='last')
per_class_df.to_csv(PER_CLASS_PATH, index=False)
print(f'Wrote {PER_CLASS_PATH.relative_to(CACHE_DIR)}')

# 5) Confusion matrix (139 x 139).
cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
cm_df.index.name = 'true_label'
cm_df.to_csv(CONF_MATRIX_PATH)
print(f'Wrote {CONF_MATRIX_PATH.relative_to(CACHE_DIR)}  ({cm.shape[0]}x{cm.shape[1]})')

# 6) Top-N most-confused pairs.
pairs = []
for i in range(num_classes):
    row_total = cm[i].sum()
    for j in range(num_classes):
        if i == j or cm[i, j] == 0:
            continue
        pairs.append({
            'true_label': class_names[i],
            'predicted_label': class_names[j],
            'count': int(cm[i, j]),
            'true_class_support': int(row_total),
            'fraction_of_true_class': float(cm[i, j] / row_total) if row_total else 0.0,
        })
pairs_df = (
    pd.DataFrame(pairs)
    .sort_values('count', ascending=False)
    .head(TOP_PAIRS_N)
    .reset_index(drop=True)
)
pairs_df.to_csv(CONFUSED_PAIRS_PATH, index=False)
print(f'Wrote {CONFUSED_PAIRS_PATH.relative_to(CACHE_DIR)}  (top {TOP_PAIRS_N})')

# 7) Console summary.
overall_top1 = float((y_pred == y_true).mean())
overall_top3 = float(np.mean([yt in t3 for yt, t3 in zip(y_true, top3_pred)]))
print('\n=== Test-set summary ===')
print(f'Test images:    {len(y_true):,}')
print(f'Classes:        {num_classes}')
print(f'Top-1 accuracy: {overall_top1:.4f}')
print(f'Top-3 accuracy: {overall_top3:.4f}')
print(f'Macro F1:       {report_dict["macro avg"]["f1-score"]:.4f}')
print(f'Weighted F1:    {report_dict["weighted avg"]["f1-score"]:.4f}')

print('\nWorst 5 classes by top-1 accuracy:')
for _, row in per_class_df.head(5).iterrows():
    print(f'  {row["crop_disease_label"]:60s}  acc={row["top1_accuracy"]:.3f}  support={row["support"]}')

print('\nTop 5 most-confused pairs:')
for _, row in pairs_df.head(5).iterrows():
    pct = row['fraction_of_true_class'] * 100
    print(f'  {row["true_label"]:40s} -> {row["predicted_label"]:40s} '
          f'{row["count"]:3d} ({pct:5.1f}% of true class)')

print(f'\nAll artifacts in: {REPORTS_DIR}')


Running inference on 9,802 test images ...
 83/307 [=======>......................] - ETA: 3:05